# **Practice: SHAP for ML models**

Hi, everyone! We are glad to see you in the second practical part of the module. Now our collection of methods has one more in it — SHAP.

Based on the knowledge from the previous lesson you remember that **SHAP** is an adaptive method, which is also why it is so frequently used. However, Shapley values have one more reason to be loved — an incredibly beautiful interpretation process.

We will work with the same dataset as in the previous module — California houses. Let us recall that every row in the dataset represents a separate residential district and contains information about the average characteristics of the houses in that district.

**The main characteristics:**

**The target variable (target):**

- `MedHouseVal` — the median value of the houses in this district, expressed in hundreds of thousands of dollars.

**The features (features):**

- `MedInc` (Median Income): the median income in the district (in tens of thousands of dollars)

- `HouseAge` (Housing Median Age): the median age of the houses in the district (in years)
- `AveRooms` (Average Rooms per Dwelling): the average number of rooms per dwelling.
- `AveBedrms` (Average Bedrooms per Dwelling): the average number of bedrooms per dwelling.
- `Population (Population):` the population of the district.
-  `AveOccup (Average Occupancy):` the average number of occupants per dwelling.
-  `Latitude (Latitude):` the geographical latitude of the district.
- `Longitude (Longitude):` the geographical longitude of the district.

Unlike the previous practical part, we will use the CatBoost model. We hope this practice will be interesting and visually pleasant for you! \

Let us get started! Enjoy the coding!

In [ ]:
#install the necessary libraries
!pip install shap catboost -q

In [ ]:
import catboost
import pandas as pd
import numpy as np
import shap
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score


data = pd.read_csv('https://github.com/SadSabrina/explainable_AI_course/raw/refs/heads/main/data/fetch_california_housing.csv',
                   index_col=0)

X, y = data.drop('target', axis=1), data['target']

rng = np.random.RandomState(0)
bin_var = pd.Series(rng.randint(0, 1, X.shape[0]), name="rnd_bin")
num_var = pd.Series(np.arange(X.shape[0]), name="rnd_num")

X_with_rnd_feat = pd.concat((X, bin_var, num_var), axis=1)

As before — let us immediately train a model on the data we have. This time, however, we skip the feature normalisation step.

**Quiz 1: Why can the normalisation step be omitted?**

In [ ]:
#split the data into train and test

X_train, X_test, y_train, y_test = train_test_split(
    X_with_rnd_feat, y, random_state=42
)

model = CatBoostRegressor(iterations=300, learning_rate=0.1, random_seed=123)
model.fit(X_train, y_train, verbose=False, plot=False)

model.best_score_

## **Analysing individual predictions with SHAP**

1. **Force plot.**

Literally, this type of plot reflects the *strength* of the influence of a specific feature value on the model prediction. For more detail the plot is coloured in blue (a decrease of the model prediction) and red (an increase of the model prediction).


**For any single observation the force plot lets you see:**
- the average value of the target feature (more precisely, its expectation) *over the original input data*
- the values of the specific variables that took part in predicting the target value for the feature
- the strength and the direction of the contribution of the specific feature values to the change of the prediction away from the average

In [ ]:
shap.initjs()
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_train)

# Visualise the predictions for the 7th object in the training data
shap.plots.force(shap_values[7, :])

The waterfall plot helps to break the force plot down to specific values.

The only downside of the `force_plot` is that it is impossible to read off the specific Shapley values. However, to remove this, we can break the `force_plot` down with the help of the `waterfall_plot`

In [ ]:
# Visualise the predictions for the 7th object in the training data
shap.plots.waterfall(shap_values[7, :], max_display=5)

Or the `bar` plot.

In [ ]:
shap.plots.bar(shap_values[7], max_display=5)

**Quiz 2: Build a similar plot for the observation with index 7 in the test dataset. After that, mark the correct statements below in bold.**

In [ ]:
shap.initjs()

# Your code here

Mark the correct statements in bold:
- **6 features took part in building the prediction for the object**
- The average value of MedInc predicted by the algorithm for X_test = 2.063
- **The model prediction for this object equals 1.64**
- Latitude led to an increase of the prediction over the base value by 33.92
- **The number of features that decreased the prediction from the base value is 4**

If you rotate the plot above by 90 degrees and stack it, you get the following picture

In [ ]:
shap.initjs()
shap.plots.force(shap_values[7:9,]) #a stacked plot for two observations

For large incoherent segments of the dataset the plot is not very informative (out of curiosity, try to get the plot for a large number of values or look at the example below). From a practical point of view, what turns out to be useful for us here is that this plot may come in handy when examining individual features.

## **Analysing the global behaviour of the model with SHAP**

In [ ]:
shap.initjs()

#Let us look at the plot for the first 100 objects of the training dataset
shap.plots.force(shap_values[:100])

By choosing individual scales on the x and y axes, we can see the main tendency of a feature's influence for all the datasets. For example, by choosing MedInc effects on the left and MedInc on the top respectively, you can see that this feature is able both to increase and to decrease the prediction away from its expectation.

**Quiz 3: Build a summary force plot for the first 100 objects from the test dataset. Mark the correct statements in bold.**

In [ ]:
shap.initjs()

#Your code here

Mark the correct statements for the selected 100 objects:
- **The feature that increases the prediction over the base value most often is Population**
- The feature that increases the prediction over the base value most often is MedInc
- **The feature that decreases it from the base value most often is Latitude**

**Quiz 4: Check whether the random feature `rnd_num` has any influence on the model? And what about `rnd_bin`?**

Next, we can try to look at whether there is an explicit pairwise interaction of features. It is called *feature interaction*.

Feature interaction is the combined effect of two or more features on the target variable. In other words, an interaction happens when the effect of one feature on the target variable depends on the value of another feature.

An example of a pairwise interaction is the dependence of the price of a house on its area. For instance, a large area may lead to a higher price, but the explicit effect of such behaviour is observed only for young houses.  

In [ ]:
# Let us create a dependence plot to examine the influence of a pair of features on the model output
shap.dependence_plot("AveRooms", shap_values.values[:100], X_train[:100], interaction_index="HouseAge")

Let us figure out the axes of the plot.
- The left scale reflects the Shapley value. Look carefully at the ranges over which the values change, so as not to draw a wrong conclusion. For example, it may seem that for older houses a small average number of rooms lowers the price, but that is not so. In fact, the older the house, the less effect the average number of rooms per house has.
- The right and the bottom scales reflect the features whose interaction we are examining.

**Quiz 5: Build a `dependence_plot` for the same features, for the first 100 objects, but based on test_explainer and test_shap_values. Select the correct statements in the trainer.**

In [ ]:
# Your code here

So, we have examined the influence of one feature and of a combination of two features on the behaviour of the model. Now let us look at the logical conclusion — the average contribution of each of them to the prediction.  

In [ ]:
# summarize the effects of all the features
shap.plots.beeswarm(shap_values)

This plot makes it possible to draw conclusions about the generalised influence of each feature. So, for example, we can see that:
- Latitude and longitude (the location), as they increase, lead to a decrease of the prediction;
- The higher the median income, the more strongly the contribution to the median value of the houses increases.
- Large values of Population have almost no influence on the model
- A large average number of bedrooms is able to influence the model a little

**Quiz 6: build a similar plot for the test data. Do the conclusions above remain valid for the test set?**

In [ ]:
#Your code here

- The median income is related almost linearly to the Shapley values (the higher the median income, the higher the Shapley value)
- The smaller the average number of occupants, the larger the Shapley value
- The random feature has a logically readable relationship (small values lead to small Shapley values).
- Latitude and longitude have similar silhouettes of influence, hence these two features are related (this is NOT how it should be)
- There is an observation where a small average number of rooms per dwelling led to an increase of the Shapley value.

Another way of presenting the summary importance is presenting it as a bar chart. Note that it takes the average value over the **absolute values of the Shapley coefficients**.

In [ ]:
# summarize the effects of all the features
shap.plots.bar(shap_values)

**Quiz 7: What information is lost when analysing the bar plot?**

And the last type of visualisation for today — the heatmap. We give it just for information, since without knowing the logic of the previous plots it is hard to read. However, based on what you have learnt, you can see that such a map is a good summary for almost all the possible conclusions from the Shapley coefficients.

In [ ]:
# visualisation as a heatmap
shap.plots.heatmap(shap_values[:100])

-